# FUNGI — Functional Unravelling of Network Geometry for Inference

**GRN sparsification pipeline.** Takes a dense LightGBM-inferred parent graph and prunes it into a
sparse, biologically plausible Gene Regulatory Network optimized for downstream perturbation prediction.

**Full pipeline context:**
```
Raw .h5ad  →  SPORE  →  CHITIN →  GuanLab (LightGBM)  →  FUNGI  →  SPECTRA
```

---
**Usage:** Run cells sequentially. All configuration is in `fungi_config.yaml`.
After completing a run, execute the **Cleanup** cell before starting a new one.

## Setup

In [ ]:
import os, yaml, warnings, sys, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*DataFrame is highly fragmented.*')

CONFIG_PATH = Path('fungi_config.yaml')
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

if cfg['runtime'].get('single_threaded_blas', True):
    for var in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
        os.environ[var] = '1'

RAW_GRAPH_PATH = Path(cfg['input']['graph_path'])
SC_DATA_PATH   = Path(cfg['input']['sc_data_path'])
OUTPUT_ROOT    = Path(cfg['output']['root_dir'])
SRC_ROOT       = Path('src')

for phase in cfg['output']['phases']:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)
Path(cfg['output']['figures_dir']).mkdir(parents=True, exist_ok=True)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f'Configuration loaded from {CONFIG_PATH}')
print(f'  Input graph  : {RAW_GRAPH_PATH.name}')
print(f'  Expression   : {SC_DATA_PATH.name}')
print(f'  Output root  : {OUTPUT_ROOT}')

## Phase 0 — Data Ingestion

Loads the parent GRN and expression data. If the GRN parquet contains experimental columns
(stability, md_score, sign_agreement, pert_efficiency), those are carried forward as a
multiplicative Bayesian gate applied in Phase 2. Standard GRNs without experimental columns
are fully supported — the gate defaults to identity.

In [ ]:
from graph_utils import load_graph

raw_G, raw_sparse_mat, experimental_df = load_graph(RAW_GRAPH_PATH)
N_GENES = raw_sparse_mat.shape[0]

print(f'Parent graph  : {N_GENES:,} nodes  |  {raw_sparse_mat.nnz:,} edges  |  density: {raw_sparse_mat.nnz / N_GENES**2:.4%}')
if experimental_df is not None:
    exp_cols = list(experimental_df.columns[2:])
    print(f'Experimental GRN detected — extra columns: {exp_cols}')
else:
    print('Standard GRN — no experimental columns (gate will be identity)')

adata = sc.read_h5ad(str(SC_DATA_PATH))
print(f'Expression data: {adata.shape[0]:,} cells  x  {adata.shape[1]:,} genes')

## Phase 1 — Diagnostic Calibration

Runs the perturbation impact analysis and probe functions to estimate biologically grounded
target bounds for each of the six topology parameters (α, Gini, S_max, Q, C, ρ).
These bounds define the **utopian loss function** that guides the graph search.

All probes derive their bounds from the expression data and parent graph alone — no external
databases or organism-specific annotations are used (dataset agnostic by design).

In [ ]:
from diagnostics import run_diagnostics

utopian_bounds, loss_weights, diagnostic_report = run_diagnostics(
    adata=adata,
    n_genes=N_GENES,
    cfg_diagnostics=cfg['diagnostics'],
    cfg_input=cfg['input'],
    raw_sparse_mat=raw_sparse_mat,
)

lam_eff = diagnostic_report['lam_eff']

diag_dir = OUTPUT_ROOT / 'phase0_diagnostics'
report_clean = {k: v for k, v in diagnostic_report.items() if not k.startswith('_')}
pd.DataFrame([report_clean]).to_json(diag_dir / 'diagnostic_report.json', indent=2)
print(f'Diagnostic report saved → {diag_dir / "diagnostic_report.json"}')

## Phase 2 — Graph Normalization and Pre-computation

Filters the parent graph to a candidate pool, computes per-edge scoring arrays, and
applies the experimental GRN gate (if present).

Weight backbone is always LightGBM Importance. Experimental signal is injected as a
multiplicative gate (`total_gate = stability_gate × md_gate`) — it modifies edge scores
within the Importance landscape but never replaces it.

- **Stability gate** — Beta-Binomial log-odds of bootstrap stability. Boosts reliably observed edges, penalizes unstable ones.
- **MD gate** — Per-source-TF rank-normalized md_score × sign_agreement. Prevents high-knockdown TFs from inflating all their edges simultaneously.

In [ ]:
from scipy.stats import rankdata
from filtering import adaptive_threshold_filter
from engine import (compute_source_quantile_weights,
                    compute_pagerank_kappa_multipliers,
                    compute_source_pert_impact,
                    build_experimental_modifiers)

G_work = adaptive_threshold_filter(
    raw_sparse_mat, target_density=cfg['prefilter']['target_density'])
gc.collect()

G_work_coo  = G_work.tocoo()
sources_raw = G_work_coo.row.copy()
targets_raw = G_work_coo.col.copy()
weights_raw = G_work_coo.data.copy()

W_ranked = rankdata(weights_raw, method='average').astype(np.float64) / len(weights_raw)
if W_ranked.max() - W_ranked.min() < 1e-6:
    print('WARNING: Degenerate weights detected — applying log1p fallback')
    W_log = np.log1p(weights_raw.astype(np.float64))
    W_arr = (np.clip(W_log / max(W_log.max(), 1e-10), 0.001, 1.0)
             if W_log.max() > 1e-12
             else np.random.default_rng(42).uniform(0.01, 1.0, len(weights_raw)))
else:
    W_arr = W_ranked

out_deg_raw = np.bincount(sources_raw, minlength=N_GENES).astype(np.float64)
D_arr       = np.log1p(out_deg_raw)[sources_raw]
sources_arr = sources_raw
targets_arr = targets_raw

print(f'Candidate pool: {len(W_arr):,} edges  (prefilter density: {cfg["prefilter"]["target_density"]*100:.0f}%)')

W_source_quantile = compute_source_quantile_weights(sources_arr, W_arr, N_GENES)

pr_cfg = cfg.get('pagerank_kappa', {})
per_gene_kappa = compute_pagerank_kappa_multipliers(
    G_work, N_GENES,
    alpha=pr_cfg.get('alpha', 0.85),
    n_iter=pr_cfg.get('n_iter', 60),
    hub_percentile=pr_cfg.get('hub_percentile', 99.0),
    hub_multiplier=pr_cfg.get('hub_multiplier', 3.0),
)
n_hubs = int((per_gene_kappa > 1.5).sum())
print(f'  PageRank hub genes (relaxed kappa cap): {n_hubs}')

impact_arr_raw = np.array(diagnostic_report.get('_impact_array', []), dtype=np.float64)
pert_labels    = np.array(diagnostic_report.get('_perturbation_labels', []))
name_to_idx    = diagnostic_report.get('_name_to_idx', {})

exp_cfg       = cfg.get('experimental_grn', {})
alpha_md      = exp_cfg.get('alpha_md', 0.5)
alpha_stab    = exp_cfg.get('alpha_stab', 0.3)
n_bootstraps  = exp_cfg.get('n_bootstraps', 20)
tau_shrinkage = exp_cfg.get('tau_shrinkage', 0.5)

total_gate, pert_efficiency_map = build_experimental_modifiers(
    experimental_df, sources_raw, targets_raw,
    list(adata.var_names), N_GENES,
    alpha_md=alpha_md, alpha_stab=alpha_stab,
    n_bootstraps=n_bootstraps, tau_shrinkage=tau_shrinkage)

if pert_efficiency_map:
    source_pert_impact = compute_source_pert_impact(
        impact_arr_raw, pert_labels, name_to_idx, N_GENES, pert_efficiency_map)
    print(f'  ψ prior: {int((source_pert_impact != 1.0).sum())} genes with non-neutral prior '
          f'(DEG + pert_efficiency blend, {len(pert_efficiency_map)} efficiency values)')
else:
    source_pert_impact = compute_source_pert_impact(
        impact_arr_raw, pert_labels, name_to_idx, N_GENES)
    print(f'  ψ prior: {int((source_pert_impact != 1.0).sum())} genes with non-neutral prior (DEG only)')

if total_gate is not None and np.any(total_gate != 1.0):
    n_above = int((total_gate > 1.0).sum())
    n_below = int((total_gate < 1.0).sum())
    print(f'  Experimental gate: {n_above:,} edges boosted, {n_below:,} penalised '
          f'(range [{total_gate.min():.3f}, {total_gate.max():.3f}])')
    print(f'    alpha_md={alpha_md}  alpha_stab={alpha_stab}  '
          f'n_bootstraps={n_bootstraps}  tau_shrinkage={tau_shrinkage}')
else:
    print('  Experimental gate: identity (standard GRN)')

md_gate_raw = total_gate

### Phase 2b — Effective Resistance Scoring
 
Computes approximate per-edge effective resistance (ER) via the
Spielman-Srivastava JL sketch.  Each edge (s→t) receives a score R_st ∈
[0.05, 1.0] reflecting how structurally irreplaceable it is for connectivity.
 
High-ER edges are bottlenecks, removing them dramatically increases the
graph's electrical resistance between their endpoints (few or no alternative
paths).  Low-ER edges are redundant, the signal can route around them.
 
The score enters the DASH kernel as a fixed multiplicative factor:
    ω(s→t) = Wq^β × exp(δ×T̃) × π_s^ψ × G_st × **R_st^η**
 
η = 0.3 (fixed, not optimized).  Max spread ≈ 1.67× between highest and
lowest ER edge — a complementary global signal that does not override the
local DASH signals.
 
Set `effective_resistance.enabled: false` in config to skip entirely.

In [ ]:
er_cfg = cfg.get('effective_resistance', {})
er_enabled = er_cfg.get('enabled', True)
 
if er_enabled:
    from effective_resistance import compute_scber_scores
 
    er_normalized, inter_mask, er_raw, er_diagnostics = compute_scber_scores(
        G_csr=G_work,           # pre-filtered CSR from adaptive_threshold_filter
        sources=sources_arr,    # same edge arrays as W_arr
        targets=targets_arr,
        cfg=er_cfg,
    )
 
    er_eta = float(er_cfg.get('eta_inter', er_cfg.get('eta', 0.20)))
 
    # Save diagnostics
    er_diag_path = OUTPUT_ROOT / 'phase2_precompute' / 'er_diagnostics.json'
    er_diag_path.parent.mkdir(parents=True, exist_ok=True)
    import json
    with open(er_diag_path, 'w') as fout:
        json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                   for k, v in er_diagnostics.items()}, fout, indent=2)
 
    mode = er_diagnostics.get('mode', 'scber')
    print(f"\\n  SCBER scoring complete (mode={mode}):")
    if mode == 'scber':
        print(f"    Communities: {er_diagnostics['n_communities']} "
              f"(Q_achieved={er_diagnostics['Q_achieved']:.3f})")
        print(f"    Inter-module edges: {er_diagnostics['n_inter']:,} "
              f"({er_diagnostics['frac_inter']*100:.1f}%) — ER boost applied")
        print(f"    Intra-module edges: {er_diagnostics['n_intra']:,} — untouched (factor=1.0)")
        print(f"    η_inter={er_eta}  bridge factor range: "
              f"[{er_diagnostics['inter_factor_min']:.3f}, 1.000]")
        print(f"    High-ER bridges (R>0.9): {er_diagnostics['n_high_er_bridges']:,}")
    else:
        print(f"    {mode} — check er_diagnostics.json for details")
else:
    er_normalized = None
    inter_mask    = None
    er_eta        = 0.20
    print("  SCBER: disabled (er_scores=None, inter_mask=None → DASH factor = 1.0)")

### Phase 2C - $\chi$ Prior + $\pi$ normalization 

In [ ]:
# ── Phase 2c: Chi prior + Pi normalization ────────────────────────────────────
from engine import compute_chi_prior, compute_source_pert_impact

# Read config
chi_cfg     = cfg.get('chi_prior', {})
chi_enabled = chi_cfg.get('enabled', True)
zeta        = float(chi_cfg.get('zeta', 0.5))

# ── 1. Chi prior (perturbation pleiotropy — covers ALL 5,024 genes) ───────────
deg_col_sums = np.array(diagnostic_report.get('_deg_col_sums', []), dtype=np.float64)

if chi_enabled and len(deg_col_sums) == N_GENES:
    chi_prior = compute_chi_prior(deg_col_sums, N_GENES, zeta=zeta)
    nonzero   = int((deg_col_sums > 0).sum())
    g         = float(np.mean(deg_col_sums[deg_col_sums > 0])) if nonzero > 0 else 0.0
    n_boosted = int((chi_prior > 1.05).sum())
    print(f"  chi prior computed (zeta={zeta}):")
    print(f"    Genes ever observed as a DEG: {nonzero:,} / {N_GENES:,}")
    print(f"    Mean col_sum (g): {g:.1f}")
    print(f"    Genes meaningfully boosted (chi > 1.05): {n_boosted:,}")
    print(f"    chi range: [{chi_prior.min():.3f}, {chi_prior.max():.3f}]")
else:
    chi_prior = None
    if not chi_enabled:
        print("  chi prior: disabled in config.")
    else:
        print("  WARNING: _deg_col_sums missing — did you update diagnostics.py?")
        print("  chi prior disabled (all edges get factor 1.0).")

# ── 2. Pi normalization (out-degree normalized perturbation impact prior) ──────
deg_out_parent = np.asarray(raw_sparse_mat.sum(axis=1)).ravel().astype(np.float64)

source_pert_impact = compute_source_pert_impact(
    impact_array        = np.array(diagnostic_report['_impact_array']),
    perturbation_labels = np.array(diagnostic_report['_perturbation_labels']),
    name_to_idx         = diagnostic_report['_name_to_idx'],
    n_genes             = N_GENES,
    pert_efficiency_map = pert_efficiency_map,
    deg_out_parent      = deg_out_parent,
)
print(f"  pi_psi (out-degree normalized):")
print(f"    source_impact range: [{source_pert_impact.min():.4f}, {source_pert_impact.max():.4f}]")

## Phase 3 — Expansive Search

Evaluates a large quasi-random Sobol sequence across the full 6D hyperparameter space
(β, δ, κ, k_core, λ, ψ). Each candidate is scored by the DASH kernel and measured
against the utopian bounds. Results are sharded to disk for crash recovery.

In [ ]:
from search import generate_sobol_samples, SearchEvaluator

es_cfg = cfg['expansive_search']
hp_cfg = cfg['hyperparameter_bounds']

sobol_params, lower_bounds, upper_bounds = generate_sobol_samples(
    n_genes=N_GENES, n_samples=es_cfg['n_samples'],
    hp_cfg=hp_cfg, seed=es_cfg['random_seed'], lam_eff=lam_eff)

pert_col   = cfg['input']['perturbation_column']
ctrl_label = cfg['input']['control_label']
pert_genes = [g for g in adata.obs[pert_col].unique() if g != ctrl_label]
gene_list  = list(adata.var_names)
perturbed_nodes = np.array(
    [gene_list.index(g) for g in pert_genes
     if g in gene_list and gene_list.index(g) < N_GENES], dtype=int)
print(f'  Perturbation targets: {len(perturbed_nodes):,} genes')

evaluator = SearchEvaluator(
    W_arr=W_arr, W_q_arr=W_source_quantile, D_arr=D_arr,
    sources_arr=sources_arr, targets_arr=targets_arr,
    n_genes=N_GENES, perturbed_nodes=perturbed_nodes,
    utopian_bounds=utopian_bounds, loss_weights=loss_weights,
    shatter_cfg=cfg['shatter'],
    per_gene_kappa=per_gene_kappa,
    source_pert_impact=source_pert_impact,
    md_gate=md_gate_raw,
    er_scores=er_normalized,
    er_eta=er_eta,
    inter_mask=inter_mask,
    chi_prior=chi_prior,           
    n_workers=es_cfg['n_workers'])


t0 = time.time()
shard_dir = str(OUTPUT_ROOT / 'phase3_expansive_search' / 'shards')
df_expansive = evaluator.evaluate(
    param_list=sobol_params, chunk_size=es_cfg['chunk_size'],
    shard_dir=shard_dir, desc='  Phase 3: Expansive search')

elapsed     = time.time() - t0
n_viable    = int((df_expansive['is_shattered'] == 0).sum())
n_shattered = len(df_expansive) - n_viable
best_loss   = (df_expansive.loc[df_expansive['is_shattered'] == 0, 'utopia_loss'].min()
               if n_viable > 0 else float('inf'))
n_zero      = int((df_expansive['utopia_loss'] <= 1e-6).sum())

print(f'\nPhase 3 complete: {len(df_expansive):,} graphs evaluated in {elapsed:.1f}s')
print(f'  Viable: {n_viable:,}  |  Shattered: {n_shattered:,}  '
      f'({n_shattered/max(len(df_expansive),1)*100:.1f}% shatter rate)')
print(f'  Best loss: {best_loss:.6f}  |  Zero-loss graphs: {n_zero:,}')

df_expansive.to_csv(
    OUTPUT_ROOT / 'phase3_expansive_search' / 'expansive_results.csv', index=False)

## Phase 4 — Spatial Niching

Clusters the top-performing graphs from Phase 3 into spatial niches in hyperparameter space,
extracting one local champion per niche as an anchor coordinate. Ensures the refinement
search starts from a diverse set of promising regions rather than one tight cluster.

In [ ]:
from niching import extract_anchors

if n_viable > 0:
    anchor_coords, anchor_losses, cluster_summary = extract_anchors(
        df_results=df_expansive,
        top_fraction=cfg['niching']['top_fraction'],
        n_clusters=cfg['niching']['n_clusters'],
        random_seed=cfg['runtime']['random_seed'])

    phase4_dir = OUTPUT_ROOT / 'phase4_niching'
    np.save(phase4_dir / 'anchor_coords.npy', anchor_coords)
    cluster_summary.to_csv(phase4_dir / 'cluster_summary.csv', index=False)

    print(f'Phase 4 complete: {len(anchor_coords)} anchor coordinates across '
          f'{cfg["niching"]["n_clusters"]} spatial niches')
    print(f'  Anchor loss range: [{anchor_losses.min():.4f}, {anchor_losses.max():.4f}]')
else:
    print('ERROR: No viable graphs from Phase 3. Cannot proceed.')

## Phase 5 — Refinement Search

ML+GMM refinement with basin-aware density search.

**Loss minimization mode:** exhaustion-based GMM+RF search minimizing utopia loss.
Switches to density mode automatically once a sufficient zero-loss pool is established.

**Density maximization mode:** DBSCAN discovers distinct zero-loss basins in 6D space.
Each basin receives a quality-proportional sample and round budget. High-lambda bias
steers each density round toward the dense frontier of the basin.
A re-detection pass scans the accumulated zero-loss pool for additional basins after
the primary search completes.

In [ ]:
from refinement import run_ml_gmm_refinement

df_refinement = None
if n_viable > 0:
    df_refinement, was_skipped = run_ml_gmm_refinement(
        df_phase3=df_expansive,
        lower=lower_bounds,
        upper=upper_bounds,
        evaluator=evaluator,
        refinement_cfg=cfg['refinement'])

    if df_refinement is not None:
        df_refinement.to_csv(
            OUTPUT_ROOT / 'phase5_refinement' / 'refinement_results.csv', index=False)
        n_ref_viable = int((df_refinement['is_shattered'] == 0).sum())
        print(f'\nRefinement results saved: {len(df_refinement):,} evaluations, '
              f'{n_ref_viable:,} viable')

## Phase 6 — Cohort Selection

Selects the champion (densest zero-loss graph, or lowest-loss if no zero-loss exists)
plus 4 topologically diverse alternates via farthest-point sampling in normalized
topology space. The cohort covers meaningfully different regulatory architectures.

In [ ]:
from refinement import select_diverse_cohort

frames = [df_expansive]
if df_refinement is not None:
    frames.append(df_refinement)
df_all = pd.concat(frames, ignore_index=True)

cohort = select_diverse_cohort(df_all, utopian_bounds, N_GENES, cohort_size=5)

print('=' * 95)
print('FUNGI — Candidate Cohort (5 topologically diverse graphs)')
print('=' * 95)
print()
print(f'{"#":>3s} {"Loss":>8s} │ {"Basin":>5s} │ {"α":>6s} {"Gini":>6s} {"S_max":>6s} '
      f'{"Q":>6s} {"C":>6s} {"ρ":>7s} │ '
      f'{"β":>5s} {"δ":>5s} {"κ":>5s} {"k_c":>5s} {"λ":>6s} {"ψ":>5s} │ {"Edges":>7s}')
print('─' * 95)

for _, row in cohort.iterrows():
    rank     = int(row['cohort_rank'])
    star     = ' ★' if row['is_champion'] else '  '
    topo_str = [f'{row[col]:6.3f}' for col in ['alpha', 'Gini', 'S_max', 'Q', 'C', 'rho']]
    basin_val = row.get('basin_idx', None)
    basin_str = f"B{int(basin_val)+1}" if basin_val is not None and pd.notna(basin_val) else "—"
    print(
        f'{rank:>2d}{star} {row["utopia_loss"]:8.4f} │ '
        f'{basin_str:>5s} │ '
        f'{" ".join(topo_str)} │ '
        f'{row["beta"]:5.2f} {row["delta"]:5.2f} {row["kappa"]:5.3f} '
        f'{row["k_core"]:5.1f} {row["lambda"]:6.2f} {row["psi"]:5.2f} │ '
        f'{int(row["n_edges"]):>7,d}'
    )

print('─' * 95)
print('★ = Champion  (densest zero-loss graph, or lowest-loss if no zero-loss exists)')
print()

champ = cohort[cohort['is_champion']].iloc[0]
print('Champion topology vs targets:')
n_passing = 0
for param, col in [('alpha','alpha'), ('gini','Gini'), ('S_max','S_max'),
                   ('Q','Q'), ('C','C'), ('rho','rho')]:
    lo, hi   = utopian_bounds[param]
    val      = champ[col]
    inside   = lo <= val <= hi
    n_passing += int(inside)
    marker   = '✓' if inside else '✗'
    print(f'  {marker} {param:>5s} = {val:.4f}  (target: [{lo:.4f}, {hi:.4f}])')
print(f'  {n_passing}/6 topology targets satisfied')

print()
print('Run configuration:')
print(f'  Stability gate : {"active (alpha_stab=" + str(alpha_stab) + ")" if experimental_df is not None and np.any(total_gate != 1.0) else "identity"}')
print(f'  MD gate        : {"active (alpha_md=" + str(alpha_md) + ")" if experimental_df is not None and np.any(total_gate > 1.0) else "identity"}')
print(f'  ψ prior        : {"DEG + pert_efficiency blend" if pert_efficiency_map else "DEG only"}')
print(f'  Sign annotation: {"yes (consensus_sign)" if experimental_df is not None and "consensus_sign" in experimental_df.columns else "no"}')

## Phase 7 — Graph Output

Reconstructs the exact edge list for each selected graph from its hyperparameter recipe
and saves to parquet for downstream use (e.g. SPECTRA).

**Set `graphs_to_output` below.** Default `[1]` = champion only.
To output alternates: `[1, 3]` or `[1, 2, 3, 4, 5]`.

Output columns: `Regulator`, `Target`, `Weight` (+ `Sign` if experimental GRN is active).

The `cohort_recipes.csv` saved here contains the six hyperparameters for each graph.
Given the same parent graph, these recipes reproduce the exact same edges deterministically.

In [ ]:
# ── USER SELECTION ─────────────────────────────────────────────────────────
# Set to [1] for champion only, or e.g. [1, 3, 5] to include alternates.
graphs_to_output = [1]
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
from engine import build_graph_from_params
import joblib as jl

gene_names = list(adata.var_names)
output_dir = OUTPUT_ROOT / 'phase6_champion'

for rank_num in graphs_to_output:
    row = cohort[cohort['cohort_rank'] == rank_num]
    if len(row) == 0:
        print(f'  WARNING: No graph with cohort_rank={rank_num}')
        continue
    row = row.iloc[0]

    params = np.array([
        row['beta'], row['delta'], row['kappa'],
        row['k_core'], row['lambda'], row['psi']
    ])

    surv_s, surv_t, surv_W = build_graph_from_params(
        params,
        evaluator.Ws, evaluator.Wqs, evaluator.Ds,
        evaluator.srcs, evaluator.tgts,
        N_GENES, perturbed_nodes,
        cfg['shatter'],
        per_gene_kappa, source_pert_impact,
        md_gate=evaluator.md_gate,
        er_scores=evaluator.er_scores,
        er_eta=evaluator.er_eta,
        inter_mask=evaluator.inter_mask,
        chi_prior=evaluator.chi_prior,  # ← new
    )

    graph_df = pd.DataFrame({
        'Regulator': [gene_names[s] for s in surv_s],
        'Target':    [gene_names[t] for t in surv_t],
        'Weight':    surv_W,
    })

    if experimental_df is not None and 'consensus_sign' in experimental_df.columns:
        src_col_exp = experimental_df.columns[0]
        tgt_col_exp = experimental_df.columns[1]
        sign_lookup = dict(zip(
            zip(experimental_df[src_col_exp], experimental_df[tgt_col_exp]),
            experimental_df['consensus_sign']))
        graph_df['Sign'] = [
            sign_lookup.get((r, t), 0)
            for r, t in zip(graph_df['Regulator'], graph_df['Target'])]
        n_signed = int((graph_df['Sign'] != 0).sum())
        print(f'  Sign annotation: {n_signed:,}/{len(graph_df):,} edges '
              f'({n_signed/len(graph_df)*100:.1f}% signed)')

    tag      = 'champion' if row['is_champion'] else f'alternate_{rank_num}'
    out_path = output_dir / f'fungi_{tag}.parquet'
    graph_df.to_parquet(out_path, index=False)

    print(f'  [{tag}] {len(graph_df):,} edges  →  {out_path}')
    print(f'    loss={row["utopia_loss"]:.6f}  '
          f'β={row["beta"]:.3f}  δ={row["delta"]:.3f}  κ={row["kappa"]:.3f}  '
          f'k_core={row["k_core"]:.1f}  λ={row["lambda"]:.2f}  ψ={row["psi"]:.3f}')

cohort.to_csv(output_dir / 'cohort_recipes.csv', index=False)

jl.dump({
    'N_GENES':                  N_GENES,
    'utopian_bounds':           utopian_bounds,
    'loss_weights':             loss_weights,
    'diagnostic_report':        report_clean,
    'champion':                 cohort[cohort['is_champion']].iloc[0].to_dict(),
    'experimental_grn_active':  experimental_df is not None,
    'stability_gate_active':    experimental_df is not None and bool(np.any(total_gate != 1.0)),
    'md_gate_active':           experimental_df is not None and bool(np.any(total_gate > 1.0)),
    'alpha_md':                 alpha_md,
    'alpha_stab':               alpha_stab,
    'pert_efficiency_genes':    len(pert_efficiency_map),
}, OUTPUT_ROOT / 'pipeline_artifacts.joblib')

print(f'\nPipeline artifacts saved → {OUTPUT_ROOT / "pipeline_artifacts.joblib"}')
print('FUNGI complete.')

## Cleanup

Deletes all intermediate phase data (shards, CSVs, diagnostics).
Preserves `phase6_champion/` (output graphs + recipes) and `pipeline_artifacts.joblib`.

**Run this cell before starting a new A/B test** to prevent shard reuse from a previous run.

In [ ]:
import shutil

intermediate_phases = [
    'phase0_diagnostics', 'phase1_ingestion', 'phase2_normalization',
    'phase3_expansive_search', 'phase4_niching', 'phase5_refinement',
]

print('Cleaning intermediate data...')
for phase_name in intermediate_phases:
    phase_dir = OUTPUT_ROOT / phase_name
    if phase_dir.exists():
        shutil.rmtree(phase_dir)
        print(f'  Deleted {phase_name}/')

for phase in cfg['output']['phases']:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)

print('\nDone. Ready for next run.')
print(f'Preserved: {OUTPUT_ROOT}/phase6_champion/  and  pipeline_artifacts.joblib')